<a href="https://colab.research.google.com/github/kassmontezuma/Capstone/blob/DeepLearning/ValidacionDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import getpass
USER = "kassmontezuma"
REPO = "Capstone"
TOKEN = getpass.getpass('Introduce tu GitHub Personal Access Token: ')

!git clone https://{TOKEN}@github.com/{USER}/{REPO}.git

%cd {REPO}
!git checkout DeepLearning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ruta del ZIP de tomografías
ZIP_PATH = "/content/drive/MyDrive/INTRUSA JIJIJI/tomografias_para_drive.zip"

# Carpeta donde descomprimiremos las tomografías
TOMOGRAFIAS_DIR = "/content/tomografias"

# Ruta del modelo entrenado
MODEL_PATH = "/content/drive/MyDrive/INTRUSA JIJIJI/Modelos_DL/models/best_ResNet18+LSTM.pth"

# Ruta del archivo remaining_candidates.csv
REMAINING_PATH = "/content/remaining_candidates.csv"

# Ruta de la lista de tomografías positivas para validación
POSITIVAS_LISTA = "/content/2_TOMOGRAFIAS_POR_PRUEBA/A_Validacion_DL_Puro/positivas.txt"

# Carpeta en Drive para guardar resultados
RESULTADOS_DRIVE = "/content/drive/MyDrive/INTRUSA JIJIJI/resultados_validacion"
os.makedirs(RESULTADOS_DRIVE, exist_ok=True)



# LIBRERIAS

In [ ]:
import os
import cv2
import json
import torch
import shutil
import zipfile
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
import SimpleITK as sitk
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split

# Descomprimir tomografías

In [ ]:
if not os.path.exists(TOMOGRAFIAS_DIR):
    os.makedirs(TOMOGRAFIAS_DIR, exist_ok=True)
    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(TOMOGRAFIAS_DIR)
        print(f"Tomografías descomprimidas en {TOMOGRAFIAS_DIR}")
    else:
        raise FileNotFoundError(f"No se encontró el ZIP en {ZIP_PATH}")
else:
    print(f"Las tomografías ya están descomprimidas en {TOMOGRAFIAS_DIR}")

In [ ]:
BASE_DIR = "C:/CAPSTONE"
os.makedirs(BASE_DIR, exist_ok=True)

carpetas = [
    "2_TOMOGRAFIAS_POR_PRUEBA/A_Validacion_DL_Puro",
    "2_TOMOGRAFIAS_POR_PRUEBA/B_GradCAM",
    "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/entrenamiento",
    "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/validacion",
    "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/prueba",
    "2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/entrenamiento",
    "2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/validacion",
    "2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/prueba"
]

for carpeta in carpetas:
    os.makedirs(os.path.join(BASE_DIR, carpeta), exist_ok=True)

print("Carpetas creadas")

Carpetas creadas


In [ ]:
remaining = pd.read_csv("remaining_candidates.csv")

# IDENTIFICAR TOMOGRAFÍAS CON NÓDULOS (positivas) y SIN NÓDULOS (negativas PURAS)
# Una tomografía es NEGATIVA PURA si TODAS sus anotaciones son class=0
tomografias_con_nodulos = set(remaining[remaining['class'] == 1]['seriesuid'].unique())

# Encontrar tomografías que NO tienen NINGUNA anotación positiva
todas_tomografias = set(remaining['seriesuid'].unique())
tomografias_sin_nodulos = todas_tomografias - tomografias_con_nodulos

# Verificar que las negativas puras NO tienen anotaciones positivas
print(f"   Tomografías con nódulos: {len(tomografias_con_nodulos)}")
print(f"   Tomografías sin nódulos (puras): {len(tomografias_sin_nodulos)}")
print(f"   Total: {len(tomografias_con_nodulos) + len(tomografias_sin_nodulos)}")

# Opcional: Verificar que no hay solapamiento
interseccion = tomografias_con_nodulos & tomografias_sin_nodulos
if interseccion:
    print(f"   ADVERTENCIA: {len(interseccion)} tomografías están en ambas listas")
    print(f"   Esto NO debería pasar. Revisa tus datos.")

   Tomografías con nódulos: 78
   Tomografías sin nódulos (puras): 704
   Total: 782


In [ ]:
def guardar_lista(tomografias, carpeta, nombre="lista.txt"):
    if not tomografias:
        print(f"   Lista vacía en {carpeta}")
        return
    ruta = os.path.join(BASE_DIR, carpeta, nombre)
    with open(ruta, 'w') as f:
        for t in tomografias:
            f.write(f"{t}\n")
    print(f"   Guardado: {len(tomografias)} tomografías en {carpeta}")

In [ ]:
positivas = list(tomografias_con_nodulos)
print(f"\nDIVISIÓN DE TOMOGRAFÍAS POSITIVAS ({len(positivas)} totales):")

# A. Validación DL Puro (60% de las positivas)
dl_puro_pos, resto_pos = train_test_split(positivas, test_size=0.4, random_state=42)
print(f"   Validación DL Puro (positivas): {len(dl_puro_pos)}")

# B. GradCAM (20 positivas como muestra)
gradcam_pos = resto_pos[:20] if len(resto_pos) >= 20 else resto_pos
resto_pos = [t for t in resto_pos if t not in gradcam_pos]
print(f"   GradCAM (positivas): {len(gradcam_pos)}")

# C/D. Sistema Híbrido (el resto)
train_pos, temp_pos = train_test_split(resto_pos, test_size=0.4, random_state=42)
val_pos, test_pos = train_test_split(temp_pos, test_size=0.5, random_state=42)
print(f"   Sistema Híbrido (positivas): {len(train_pos)} train + {len(val_pos)} val + {len(test_pos)} test")


DIVISIÓN DE TOMOGRAFÍAS POSITIVAS (78 totales):
   Validación DL Puro (positivas): 46
   GradCAM (positivas): 20
   Sistema Híbrido (positivas): 7 train + 2 val + 3 test


In [ ]:
negativas = list(tomografias_sin_nodulos)
print(f"\nDIVISIÓN DE TOMOGRAFÍAS NEGATIVAS PURAS ({len(negativas)} totales):")

# A. Validación DL Puro (misma proporción que positivas)
dl_puro_neg, resto_neg = train_test_split(negativas, test_size=0.4, random_state=42)
print(f"   Validación DL Puro (negativas): {len(dl_puro_neg)}")

# B. GradCAM (20 negativas como muestra)
gradcam_neg = resto_neg[:20] if len(resto_neg) >= 20 else resto_neg
resto_neg = [t for t in resto_neg if t not in gradcam_neg]
print(f"   GradCAM (negativas): {len(gradcam_neg)}")

# C/D. Sistema Híbrido (el resto)
train_neg, temp_neg = train_test_split(resto_neg, test_size=0.4, random_state=42)
val_neg, test_neg = train_test_split(temp_neg, test_size=0.5, random_state=42)
print(f"   Sistema Híbrido (negativas): {len(train_neg)} train + {len(val_neg)} val + {len(test_neg)} test")


DIVISIÓN DE TOMOGRAFÍAS NEGATIVAS PURAS (704 totales):
   Validación DL Puro (negativas): 422
   GradCAM (negativas): 20
   Sistema Híbrido (negativas): 157 train + 52 val + 53 test


In [ ]:
print("\nVERIFICANDO INTEGRIDAD DE SPLITS:")

def verificar_solapamiento(lista1, lista2, nombre1, nombre2):
    solapamiento = set(lista1) & set(lista2)
    if solapamiento:
        print(f"   ERROR: {nombre1} y {nombre2} comparten {len(solapamiento)} tomografías")
        return False
    else:
        print(f"   OK: {nombre1} y {nombre2} no se solapan")
        return True

# Verificar que no hay solapamiento entre grupos de positivas
verificar_solapamiento(dl_puro_pos, gradcam_pos, "Validación_Pos", "GradCAM_Pos")
verificar_solapamiento(dl_puro_pos, train_pos, "Validación_Pos", "Train_Pos")
verificar_solapamiento(dl_puro_pos, val_pos, "Validación_Pos", "Val_Pos")
verificar_solapamiento(dl_puro_pos, test_pos, "Validación_Pos", "Test_Pos")
verificar_solapamiento(gradcam_pos, train_pos, "GradCAM_Pos", "Train_Pos")
verificar_solapamiento(train_pos, val_pos, "Train_Pos", "Val_Pos")
verificar_solapamiento(train_pos, test_pos, "Train_Pos", "Test_Pos")

print("-" * 50)

verificar_solapamiento(dl_puro_neg, gradcam_neg, "Validación_Neg", "GradCAM_Neg")
verificar_solapamiento(dl_puro_neg, train_neg, "Validación_Neg", "Train_Neg")
verificar_solapamiento(train_neg, val_neg, "Train_Neg", "Val_Neg")
verificar_solapamiento(train_neg, test_neg, "Train_Neg", "Test_Neg")


VERIFICANDO INTEGRIDAD DE SPLITS:
   OK: Validación_Pos y GradCAM_Pos no se solapan
   OK: Validación_Pos y Train_Pos no se solapan
   OK: Validación_Pos y Val_Pos no se solapan
   OK: Validación_Pos y Test_Pos no se solapan
   OK: GradCAM_Pos y Train_Pos no se solapan
   OK: Train_Pos y Val_Pos no se solapan
   OK: Train_Pos y Test_Pos no se solapan
--------------------------------------------------
   OK: Validación_Neg y GradCAM_Neg no se solapan
   OK: Validación_Neg y Train_Neg no se solapan
   OK: Train_Neg y Val_Neg no se solapan
   OK: Train_Neg y Test_Neg no se solapan


True

In [ ]:
# Validación DL Puro
guardar_lista(dl_puro_pos, "2_TOMOGRAFIAS_POR_PRUEBA/A_Validacion_DL_Puro", "positivas.txt")
guardar_lista(dl_puro_neg, "2_TOMOGRAFIAS_POR_PRUEBA/A_Validacion_DL_Puro", "negativas.txt")

# GradCAM
guardar_lista(gradcam_pos, "2_TOMOGRAFIAS_POR_PRUEBA/B_GradCAM", "positivas.txt")
guardar_lista(gradcam_neg, "2_TOMOGRAFIAS_POR_PRUEBA/B_GradCAM", "negativas.txt")

# Sistema Híbrido DL
guardar_lista(train_pos, "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/entrenamiento", "positivas.txt")
guardar_lista(train_neg, "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/entrenamiento", "negativas.txt")
guardar_lista(val_pos, "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/validacion", "positivas.txt")
guardar_lista(val_neg, "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/validacion", "negativas.txt")
guardar_lista(test_pos, "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/prueba", "positivas.txt")
guardar_lista(test_neg, "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/prueba", "negativas.txt")

# Sistema Híbrido DL+ML (mismas listas que DL)
guardar_lista(train_pos, "2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/entrenamiento", "positivas.txt")
guardar_lista(train_neg, "2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/entrenamiento", "negativas.txt")
guardar_lista(val_pos, "2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/validacion", "positivas.txt")
guardar_lista(val_neg, "2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/validacion", "negativas.txt")
guardar_lista(test_pos, "2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/prueba", "positivas.txt")
guardar_lista(test_neg, "2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/prueba", "negativas.txt")

   Guardado: 46 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/A_Validacion_DL_Puro
   Guardado: 422 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/A_Validacion_DL_Puro
   Guardado: 20 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/B_GradCAM
   Guardado: 20 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/B_GradCAM
   Guardado: 7 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/entrenamiento
   Guardado: 157 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/entrenamiento
   Guardado: 2 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/validacion
   Guardado: 52 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/validacion
   Guardado: 3 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/prueba
   Guardado: 53 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/prueba
   Guardado: 7 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/entrenamiento
   Guardado: 157 tomografías en 2_TOMOGRAFIAS_POR_PRUEBA/D_Sistema_Hibrido_DL_ML/entrenamiento
 

# Empaquetar tomografia

In [ ]:
DATA_PATH = "./data"

def cargar_tomografias_necesarias():
    tomografias = set()

    carpetas = [
        "2_TOMOGRAFIAS_POR_PRUEBA/A_Validacion_DL_Puro",
        "2_TOMOGRAFIAS_POR_PRUEBA/B_GradCAM",
        "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/entrenamiento",
        "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/validacion",
        "2_TOMOGRAFIAS_POR_PRUEBA/C_Sistema_Hibrido_DL/prueba",
    ]

    for carpeta in carpetas:
        # Leer positivas
        path_pos = os.path.join(BASE_DIR, carpeta, "positivas.txt")
        if os.path.exists(path_pos):
            with open(path_pos, 'r') as f:
                for line in f:
                    tomografias.add(line.strip())

        # Leer negativas
        path_neg = os.path.join(BASE_DIR, carpeta, "negativas.txt")
        if os.path.exists(path_neg):
            with open(path_neg, 'r') as f:
                for line in f:
                    tomografias.add(line.strip())

    return list(tomografias)

# Obtener lista de tomografías necesarias
tomografias_necesarias = cargar_tomografias_necesarias()
print(f" Total tomografías a subir: {len(tomografias_necesarias)}")
print(f"   Esto es {len(tomografias_necesarias)/782*100:.1f}% del total de 782")

 Total tomografías a subir: 782
   Esto es 100.0% del total de 782


In [ ]:
# ============================================
# CREAR CARPETA TEMPORAL PARA SUBIR
# ============================================
TEMP_DIR = "C:/CAPSTONE/temp_upload"
os.makedirs(TEMP_DIR, exist_ok=True)

# Función para buscar una tomografía en los subsets
def encontrar_tomografia(seriesuid, data_path):
    for subset in range(10):
        mhd_path = os.path.join(data_path, f"subset{subset}", f"{seriesuid}.mhd")
        raw_path = os.path.join(data_path, f"subset{subset}", f"{seriesuid}.raw")
        if os.path.exists(mhd_path):
            return mhd_path, raw_path
    return None, None

# ============================================
# COPIAR TOMOGRAFÍAS A CARPETA TEMPORAL
# ============================================
print("\nCopiando tomografías a carpeta temporal...")
copiadas = 0
no_encontradas = []

for seriesuid in tqdm(tomografias_necesarias, desc="Copiando"):
    mhd_path, raw_path = encontrar_tomografia(seriesuid, DATA_PATH)

    if mhd_path:
        # Copiar .mhd
        dest_mhd = os.path.join(TEMP_DIR, f"{seriesuid}.mhd")
        shutil.copy2(mhd_path, dest_mhd)

        # Copiar .raw si existe
        if raw_path and os.path.exists(raw_path):
            dest_raw = os.path.join(TEMP_DIR, f"{seriesuid}.raw")
            shutil.copy2(raw_path, dest_raw)

        copiadas += 1
    else:
        no_encontradas.append(seriesuid)

print(f"\nTomografías copiadas: {copiadas}")
print(f"No encontradas: {len(no_encontradas)}")

if no_encontradas:
    print("   Primeras 5 no encontradas:")
    for t in no_encontradas[:5]:
        print(f"      - {t}")

# ============================================
# CREAR ARCHIVO ZIP PARA SUBIR
# ============================================
import zipfile

zip_path = "C:/CAPSTONE/tomografias_para_drive.zip"
print(f"\n Creando archivo ZIP: {zip_path}")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in tqdm(os.listdir(TEMP_DIR), desc="Comprimiendo"):
        file_path = os.path.join(TEMP_DIR, file)
        zipf.write(file_path, file)

# Calcular tamaño del ZIP
tamano_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f" ZIP creado: {tamano_mb:.2f} MB")

# ============================================
# GUARDAR LISTA DE TOMOGRAFÍAS SUBIDAS
# ============================================
with open(os.path.join(BASE_DIR, "tomografias_subidas.json"), "w") as f:
    json.dump({
        "total": copiadas,
        "seriesuids": tomografias_necesarias,
        "no_encontradas": no_encontradas
    }, f, indent=4)

print(f"\n Lista guardada en: {BASE_DIR}/tomografias_subidas.json")



Copiando tomografías a carpeta temporal...


Copiando: 100%|██████████| 782/782 [02:36<00:00,  4.99it/s]



Tomografías copiadas: 782
No encontradas: 0

 Creando archivo ZIP: C:/CAPSTONE/tomografias_para_drive.zip


Comprimiendo: 100%|██████████| 1564/1564 [1:17:39<00:00,  2.98s/it]

 ZIP creado: 49882.78 MB

 Lista guardada en: C:/CAPSTONE/tomografias_subidas.json


# colab

In [ ]:
# ============================================
# PASO 2: Definir la arquitectura del modelo
# ============================================
class ResNet18LSTM(nn.Module):
    def __init__(self, num_classes=1, lstm_hidden=256, num_layers=2, dropout=0.2):
        super(ResNet18LSTM, self).__init__()
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()
        self.lstm = nn.LSTM(in_features, lstm_hidden, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        batch_size, seq_len, C, H, W = x.shape
        x = x.view(batch_size * seq_len, C, H, W)
        features = self.resnet(x)
        features = features.view(batch_size, seq_len, -1)
        lstm_out, _ = self.lstm(features)
        last_output = lstm_out[:, -1, :]
        output = self.fc(last_output)
        return output.squeeze(1)

# ============================================
# PASO 3: Cargar modelo entrenado
# ============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

model = ResNet18LSTM().to(device)
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Modelo cargado correctamente")

# ============================================
# PASO 4: Función para extraer patch 3D de una tomografía (con clipping HU y normalización)
# ============================================
def extraer_patch_desde_tomografia(mhd_path, coordX, coordY, coordZ,
                                    hu_min=-1000, hu_max=400, patch_size=64):

    # Leer imagen
    sitk_img = sitk.ReadImage(mhd_path)
    volume = sitk.GetArrayFromImage(sitk_img)  # (z,y,x)
    origin = sitk_img.GetOrigin()  # (x,y,z) en mm
    spacing = sitk_img.GetSpacing()  # (x,y,z) en mm

    # Coordenadas voxel (cuidado: volume se indexa como z,y,x)
    voxelX = int((coordX - origin[0]) / spacing[0])
    voxelY = int((coordY - origin[1]) / spacing[1])
    voxelZ = int((coordZ - origin[2]) / spacing[2])

    half = patch_size // 2
    z_start = max(0, voxelZ - half)
    z_end = min(volume.shape[0], voxelZ + half)
    y_start = max(0, voxelY - half)
    y_end = min(volume.shape[1], voxelY + half)
    x_start = max(0, voxelX - half)
    x_end = min(volume.shape[2], voxelX + half)

    patch_hu = volume[z_start:z_end, y_start:y_end, x_start:x_end].astype(np.float32)

    # Clipping HU
    patch_clipped = np.clip(patch_hu, hu_min, hu_max)

    # Normalización a [0,1]
    patch_norm = (patch_clipped - hu_min) / (hu_max - hu_min)

    # Si el patch es más pequeño, rellenar con ceros
    if patch_norm.shape != (patch_size, patch_size, patch_size):
        temp = np.zeros((patch_size, patch_size, patch_size), dtype=np.float32)
        temp[:patch_norm.shape[0], :patch_norm.shape[1], :patch_norm.shape[2]] = patch_norm
        patch_norm = temp

    return patch_norm  # valores en [0,1]

# ============================================
# PASO 5: Función patch_a_secuencia
# ============================================
def patch_a_secuencia(patch, seq_len=16, target_size=(224, 224)):
    """
    Convierte patch 3D (64,64,64) en tensor (seq_len,3,224,224)
    usando la misma normalización ImageNet que en el dataset.
    """
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    z_center = patch.shape[0] // 2
    total_slices = patch.shape[0]
    half_seq = seq_len // 2
    start_idx = max(0, z_center - half_seq)
    end_idx = min(total_slices, z_center + half_seq)
    indices = np.linspace(start_idx, end_idx - 1, seq_len).astype(int)

    sequence = []
    for z in indices:
        axial_slice = patch[z, :, :]  # (64,64)
        axial_uint8 = (axial_slice * 255).astype(np.uint8)
        axial_resized = cv2.resize(axial_uint8, target_size, interpolation=cv2.INTER_LINEAR)
        axial_norm = axial_resized / 255.0
        slice_tensor = torch.from_numpy(axial_norm).float().unsqueeze(0)
        slice_tensor = slice_tensor.repeat(3, 1, 1)
        slice_tensor = normalize(slice_tensor)
        sequence.append(slice_tensor)
    return torch.stack(sequence, dim=0)  # (seq_len, 3, H, W)

# ============================================
# PASO 6: 5 tomografía positiva y extraer el nódulo real
# ============================================
remaining = pd.read_csv(REMAINING_PATH)

with open(POSITIVAS_LISTA, 'r') as f:
    tomografias_positivas = [line.strip() for line in f if line.strip()]

resultados = []

for idx, seriesuid in enumerate(tomografias_positivas[:5]):
    print(f"\n[{idx+1}/5] Probando con tomografía: {seriesuid}")

    # Buscar anotación positiva
    fila = remaining[(remaining['seriesuid'] == seriesuid) & (remaining['class'] == 1)]
    if fila.empty:
        print(f"No se encontró anotación positiva para {seriesuid}")
        continue

    coordX = fila.iloc[0]['coordX']
    coordY = fila.iloc[0]['coordY']
    coordZ = fila.iloc[0]['coordZ']

    mhd_path = os.path.join(TOMOGRAFIAS_DIR, f"{seriesuid}.mhd")
    if not os.path.exists(mhd_path):
        print(f"Archivo .mhd no encontrado: {mhd_path}")
        continue

    try:
        patch = extraer_patch_desde_tomografia(mhd_path, coordX, coordY, coordZ)
        seq = patch_a_secuencia(patch)
        input_tensor = seq.unsqueeze(0).to(device)
        with torch.no_grad():
            logit = model(input_tensor)
            prob = torch.sigmoid(logit).item()

        resultados.append({
            "seriesuid": seriesuid,
            "probabilidad": prob,
            "coordX": coordX,
            "coordY": coordY,
            "coordZ": coordZ
        })
        print(f"Probabilidad: {prob:.4f} ({prob*100:.2f}%)")
    except Exception as e:
        print(f"Error: {e}")

# Guardar resultados en Drive
df_resultados = pd.DataFrame(resultados)
csv_path = os.path.join(RESULTADOS_DRIVE, "validacion_primeras_5_positivas.csv")
df_resultados.to_csv(csv_path, index=False)
print(f"\nResultados guardados en: {csv_path}")
print(df_resultados)